# Celda 1 — generar JSON consolidado + control de archivos

In [0]:
import zipfile
import os
import pandas as pd
import json
import io
import re
from datetime import datetime

source_path = "/Volumes/weather/raw/ana_volume/historical/"
output_json = "/Volumes/weather/raw/ana_volume/json/Histo/ana_pluvio_daily.json"
control_file = "/Volumes/weather/raw/ana_volume/json/Histo/ana_processed_files.json"

files = sorted([f for f in os.listdir(source_path) if f.endswith(".zip")])

processed=set()
if os.path.exists(control_file):
    with open(control_file) as f:
        processed=set(json.load(f))

records=[]


def find_header_line(lines):
    for i,l in enumerate(lines):
        if l.startswith("EstacaoCodigo"):
            return i
    return None


for file in files:

    if file in processed:
        continue

    full_path=os.path.join(source_path,file)

    try:

        with zipfile.ZipFile(full_path) as z:

            csv_files=[x for x in z.namelist() if x.endswith(".csv")]

            if not csv_files:
                print("no csv:",file)
                continue

            with z.open(csv_files[0]) as f:

                content=f.read().decode("latin1")
                lines=content.splitlines()

                header=find_header_line(lines)

                if header is None:
                    print("header not found:",file)
                    continue

                csv_text="\n".join(lines[header:])

                df=pd.read_csv(
                    io.StringIO(csv_text),
                    sep=";",
                    dtype=str,
                    engine="python"
                )


        df["NivelConsistencia"]=df["NivelConsistencia"].astype(float)

        # seleccionar mejor consistencia
        df=df.sort_values("NivelConsistencia",ascending=False)
        df=df.drop_duplicates(subset=["EstacaoCodigo","Data"],keep="first")

        import re

        chuva_cols = [c for c in df.columns if re.match(r"^Chuva\d{2}$", c)]

        for _, row in df.iterrows():

            base_date = datetime.strptime(row["Data"], "%d/%m/%Y")

            for c in chuva_cols:

                day = int(c[-2:])   # toma los dos últimos caracteres

                rain = row[c]

                if pd.isna(rain) or rain == "":
                    continue

                try:
                    date = base_date.replace(day=day)
                except:
                    continue

        rain = row[c]

        if pd.isna(rain) or rain == "":
            rain_value = None
            rain_status = "0"
        else:
            rain_value = str(rain)
            rain_status = "0"

        records.append({
            "Chuva_Adotada": rain_value,
            "Chuva_Adotada_Status": rain_status,
            "Cota_Adotada": None,
            "Cota_Adotada_Status": "0",
            "Data_Atualizacao": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            "Data_Hora_Medicao": date.strftime("%Y-%m-%d 12:00:00"),
            "Vazao_Adotada": None,
            "Vazao_Adotada_Status": "0",
            "codigoestacao": str(row["EstacaoCodigo"])
        })

        processed.add(file)
        print("OK:",file)

    except Exception as e:
        print("ERROR:",file,e)



if records:

    mode="a" if os.path.exists(output_json) else "w"

    with open(output_json,mode) as f:
        for r in records:
            f.write(json.dumps(r)+"\n")


with open(control_file,"w") as f:
    json.dump(list(processed),f)

print("procesados:",len(processed),"/",len(files))

# Celda 2 — carga idempotente a Delta

In [0]:
from pyspark.sql.functions import col, to_timestamp
from delta.tables import DeltaTable

json_path = "/Volumes/weather/raw/ana_volume/json/Histo/ana_pluvio_daily.json"

df = spark.read.json(json_path)

columns = [
    "Chuva_Adotada",
    "Chuva_Adotada_Status",
    "Cota_Adotada",
    "Cota_Adotada_Status",
    "Data_Atualizacao",
    "Data_Hora_Medicao",
    "Vazao_Adotada",
    "Vazao_Adotada_Status",
    "codigoestacao"
]

df = (
    df.select(columns)
    .withColumn("Data_Hora_Medicao", to_timestamp("Data_Hora_Medicao"))
    .withColumn("Data_Atualizacao", to_timestamp("Data_Atualizacao"))
)

target_table = "weather.bronze.ana_rio_uruguai"

try:
    # If table exists, perform merge
    delta = DeltaTable.forName(spark, target_table)
    (
        delta.alias("t")
        .merge(
            df.alias("s"),
            """
            t.codigoestacao = s.codigoestacao
            AND t.Data_Hora_Medicao = s.Data_Hora_Medicao
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
except Exception:
    # If table does not exist, create it
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

print("carga finalizada")